# 🚀 Sprint 5 — Full Pipeline, QA & Documentation
**Project:** Autonomous Medical Research & Patient Triage Assistant
**Day:** 5 of 5 | **Sprint Goal:** Complete end-to-end pipeline + QA + AGENTS.md

## What we build today
```
Full Pipeline Test:
  Telegram/WhatsApp Message
    -> Sprint 1: Validate + Mode Detection + Red Flag Screen
    -> Sprint 2: PubMed + WHO/NHIS Research + Pinecone Storage
    -> Sprint 3: RAG Retrieve + Cohere Rerank + LangGraph Triage
    -> Sprint 4: Report Generation + Nutrition + Delivery
```

## Sprint 5 User Stories
> **US-06:** AGENTS.md documents all agent instructions and skills
> **US-07:** Live demo runs full pipeline end-to-end without errors

## Definition of Done
- Full pipeline runs for 3 different conditions
- Safety disclaimer on 100% of responses
- AGENTS.md committed
- skills/ directory created
- Demo runs cleanly

In [ ]:
import os, json, time, uuid, re, pathlib, requests, difflib
from datetime import datetime, timezone
from typing import TypedDict, Optional, List, Dict
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone, ServerlessSpec
import cohere
from langgraph.graph import StateGraph, END

load_dotenv()
OPENAI_API_KEY   = os.getenv('OPENAI_API_KEY', '')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY', '')
COHERE_API_KEY   = os.getenv('COHERE_API_KEY', '')
TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN', '')
TELEGRAM_CHAT_ID   = os.getenv('TELEGRAM_CHAT_ID', '')

assert OPENAI_API_KEY,   'Set OPENAI_API_KEY'
assert PINECONE_API_KEY, 'Set PINECONE_API_KEY'
assert COHERE_API_KEY,   'Set COHERE_API_KEY'

openai_client  = OpenAI(api_key=OPENAI_API_KEY)
pc             = Pinecone(api_key=PINECONE_API_KEY)
cohere_client  = cohere.Client(COHERE_API_KEY)

PINECONE_INDEX_NAME = 'medical-triage-agent'
EMBED_MODEL         = 'text-embedding-3-small'
RESEARCH_MODEL      = 'gpt-4o-mini'
REPORT_MODEL        = 'gpt-4o'
PUBMED_BASE_URL     = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

pinecone_index = pc.Index(PINECONE_INDEX_NAME)
print('All clients ready')
print(f'Telegram: {"configured" if TELEGRAM_BOT_TOKEN else "not set (optional)"}')

In [ ]:
class MedicalAgentState(TypedDict):
    session_id:        str
    ticket_id:         str
    initiated_at:      str
    channel:           str
    chat_id:           str
    user_mode:         str
    raw_message:       str
    corrected_message: Optional[str]
    symptoms:          List[str]
    duration:          Optional[str]
    age:               Optional[str]
    medications:       List[str]
    status:            str
    errors:            List[str]
    raw_research:      Optional[str]
    research_chunks:   Optional[List[str]]
    pinecone_ids:      Optional[List[str]]
    retrieved_chunks:  Optional[List[str]]
    reranked_chunks:   Optional[List[str]]
    urgency_level:     Optional[str]
    urgency_score:     Optional[int]
    differential:      Optional[List[str]]
    red_flags:         Optional[List[str]]
    retry_count:       int
    triage_report:     Optional[Dict[str, str]]
    nutrition_advice:  Optional[str]
    home_remedies:     Optional[str]
    follow_up_sent:    bool
    report_ready:      bool
    workflow_path:     List[str]

def create_state(message: str, channel: str = 'telegram', mode_hint: str = '') -> MedicalAgentState:
    return MedicalAgentState(
        session_id=str(uuid.uuid4()), ticket_id='', initiated_at='',
        channel=channel, chat_id=TELEGRAM_CHAT_ID or 'test',
        user_mode='patient', raw_message=message, corrected_message=None,
        symptoms=[], duration=None, age=None, medications=[],
        status='pending', errors=[],
        raw_research=None, research_chunks=None, pinecone_ids=None,
        retrieved_chunks=None, reranked_chunks=None,
        urgency_level=None, urgency_score=None,
        differential=None, red_flags=None, retry_count=0,
        triage_report=None, nutrition_advice=None,
        home_remedies=None, follow_up_sent=False,
        report_ready=False, workflow_path=[]
    )

print('State schema and helper ready')

## Sprint 1 Nodes (full pipeline)

In [ ]:
CLINICIAN_KEYWORDS = [
    'i am a doctor', 'i am a nurse', 'i am a clinician', 'i am a pharmacist',
    'my patient', 'the patient', 'presenting with', 'chief complaint',
    'differential', 'treatment protocol', 'drug of choice', 'nhis provider'
]
RED_FLAG_SYMPTOMS = {
    'cardiac':      ['chest pain', 'cannot breathe', 'shortness of breath'],
    'neurological': ['seizure', 'convulsion', 'fits', 'unconscious', 'stroke'],
    'haemorrhage':  ['vomiting blood', 'blood in stool', 'bleeding heavily'],
    'sepsis':       ['temperature 40', 'not responding', 'shaking badly'],
}
NIGERIAN_NUTRITION = {
    'malaria':      {'eat': ['Bitter leaf soup', 'Citrus fruits', 'Plenty water', 'Light pap'], 'avoid_during': ['Heavy oily food', 'Alcohol', 'Spicy food until you feel better'], 'tip': 'Drink 2-3 litres of water daily.'},
    'typhoid':      {'eat': ['Light pap', 'Oats', 'Boiled yam', 'Plenty clean water'], 'avoid_during': ['Pepper', 'Fried food', 'Street food', 'Raw vegetables'], 'tip': 'Eat small portions. Wash hands before eating.'},
    'anaemia':      {'eat': ['Ugu (pumpkin leaves)', 'Liver', 'Beans', 'Ofada rice', 'Watermelon'], 'avoid_during': ['Tea with meals'], 'avoid_long_term': ['Too much alcohol'], 'tip': 'Eat vitamin C foods with iron-rich foods.'},
    'hypertension': {'eat': ['Garden egg', 'Fish (grilled)', 'Oats', 'Banana'], 'avoid_long_term': ['Salt', 'Seasoning cubes', 'Fried foods', 'Red meat', 'Sugary drinks', 'Alcohol'], 'tip': 'Cook without Maggi. Use herbs for flavour.'},
    'diabetes':     {'eat': ['Tiger nuts', 'Unripe plantain', 'Garden egg', 'Vegetables'], 'avoid_long_term': ['Eba/fufu large portions', 'White bread', 'Sugary drinks', 'Puff puff', 'Sweet chin chin'], 'tip': 'Eat at regular times.'},
    'default':      {'eat': ['Plenty water', 'Fruits and vegetables', 'Light easily digestible foods'], 'avoid_during': ['Alcohol', 'Very oily food'], 'tip': 'Rest and stay hydrated.'},
}

SPELLING_CORRECTIONS = {
    'feaver': 'fever', 'fiever': 'fever', 'hedache': 'headache', 'headake': 'headache',
    'maleria': 'malaria', 'malaeria': 'malaria', 'tyfoid': 'typhoid', 'typoid': 'typhoid',
    'diarhoea': 'diarrhea', 'diareah': 'diarrhea', 'nusea': 'nausea', 'vomitting': 'vomiting',
    'dizzyness': 'dizziness', 'breething': 'breathing', 'shorness': 'shortness',
    'abdomnial': 'abdominal', 'stomack': 'stomach'
}
SPELLING_PHRASES = {
    r'\bshort of breath\b': 'shortness of breath',
    r'\bchest pains\b': 'chest pain',
    r'\bbody ache\b': 'body ache',
}
SPELLING_VOCAB = sorted({
    'fever', 'headache', 'malaria', 'typhoid', 'diarrhea', 'nausea', 'vomiting',
    'dizziness', 'breathing', 'shortness', 'stomach', 'abdominal', 'chills',
    'weakness', 'fatigue', 'cough', 'wheezing', 'pain', 'rash', 'sweating',
    'body', 'ache', 'thirst', 'urinating', 'weight', 'loss', 'blurred',
    'vision', 'chest', 'convulsion', 'seizure', 'unconscious', 'stroke',
})

def detect_user_mode(msg: str) -> str:
    m = msg.lower()
    return 'clinician' if any(k in m for k in CLINICIAN_KEYWORDS) else 'patient'

def normalize_message_text(msg: str) -> str:
    text = msg
    for pattern, replacement in SPELLING_PHRASES.items():
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    def replace_word(match):
        word = match.group(0)
        lowered = word.lower()
        if lowered in SPELLING_CORRECTIONS:
            replacement = SPELLING_CORRECTIONS[lowered]
        elif len(lowered) < 5 or not lowered.isalpha():
            return word
        elif lowered in SPELLING_VOCAB:
            return word
        else:
            close = difflib.get_close_matches(lowered, SPELLING_VOCAB, n=1, cutoff=0.9)
            if not close:
                return word
            replacement = close[0]
        if word.isupper():
            return replacement.upper()
        if word[:1].isupper():
            return replacement.capitalize()
        return replacement

    return re.sub(r'\b[A-Za-z]{4,}\b', replace_word, text)

def llm_correct_message_text(msg: str) -> str:
    prompt = (
        'Correct spelling and obvious typos in this medical triage message.\n'
        'Rules:\n'
        '- Preserve the meaning exactly.\n'
        '- Do not add new symptoms, diagnoses, or advice.\n'
        '- Keep medication names, doses, numbers, and names unchanged.\n'
        '- Return only the corrected message text.\n\n'
        f'Message: {msg}'
    )
    try:
        resp = openai_client.chat.completions.create(
            model=RESEARCH_MODEL,
            temperature=0,
            messages=[
                {'role': 'system', 'content': 'You are a careful spelling corrector for medical text.'},
                {'role': 'user', 'content': prompt},
            ],
        )
        corrected = (resp.choices[0].message.content or '').strip()
        return corrected or msg
    except Exception as e:
        print(f'[PARSE] LLM spelling correction failed: {e}')
        return msg

def correct_message_for_parsing(msg: str) -> str:
    return normalize_message_text(llm_correct_message_text(msg))

def extract_symptoms(msg: str) -> dict:
    dur = re.search(r'(\d+\s*(?:day|days|week|weeks|hour|hours))', msg, re.IGNORECASE)
    age = re.search(r'(\d+)\s*(?:year|yr|years old)', msg, re.IGNORECASE)
    syms = [s.strip() for s in re.split(r'[,;]|\band\b', msg) if len(s.strip()) > 3]
    return {'symptoms': syms[:10], 'duration': dur.group(1) if dur else None,
            'age': age.group(1)+' years' if age else None, 'medications': []}

def detect_red_flags(msg: str) -> List[str]:
    m = msg.lower()
    return [f'{cat}: {kw}' for cat, kws in RED_FLAG_SYMPTOMS.items() for kw in kws if kw in m][:3]

def get_nutrition_advice(symptoms: List[str], differential: List[str]) -> str:
    all_text = ' '.join(symptoms + differential).lower()
    for cond, data in NIGERIAN_NUTRITION.items():
        if cond != 'default' and cond in all_text:
            parts = ['FOODS TO EAT:\n' + '\n'.join([f'  - {f}' for f in data['eat']])]
            if data.get('avoid_during'):
                parts.append('FOODS TO AVOID DURING ILLNESS:\n' + '\n'.join([f'  - {f}' for f in data['avoid_during']]))
            if data.get('avoid_long_term'):
                parts.append('FOODS TO AVOID LONG-TERM:\n' + '\n'.join([f'  - {f}' for f in data['avoid_long_term']]))
            parts.append(f'TIP: {data["tip"]}')
            return '\n\n'.join(parts)
    d = NIGERIAN_NUTRITION['default']
    parts = ['GUIDANCE:\n' + '\n'.join([f'  - {f}' for f in d['eat']])]
    if d.get('avoid_during'):
        parts.append('FOODS TO AVOID DURING ILLNESS:\n' + '\n'.join([f'  - {f}' for f in d['avoid_during']]))
    if d.get('avoid_long_term'):
        parts.append('FOODS TO AVOID LONG-TERM:\n' + '\n'.join([f'  - {f}' for f in d['avoid_long_term']]))
    parts.append(f'TIP: {d["tip"]}')
    return '\n\n'.join(parts)

print('Sprint 1 helpers loaded')

## All Pipeline Nodes

In [ ]:
# ── Sprint 1 nodes ────────────────────────────────────────────────────────────
def validate_input_node(state):
    raw = state.get('raw_message', '').strip()
    if not raw or len(raw) < 5:
        return {**state, 'status': 'error_invalid_input',
                'errors': ['Message too short'],
                'workflow_path': state.get('workflow_path', []) + ['validate']}
    now = datetime.now(timezone.utc)
    tid = f'MED-{now.strftime("%Y%m%d")}-{str(uuid.uuid4())[:8].upper()}'
    print(f'[VALIDATE] Ticket: {tid}')
    return {**state, 'ticket_id': tid, 'initiated_at': now.isoformat(),
            'status': 'validated', 'workflow_path': state.get('workflow_path', []) + ['validate']}

def parse_input_node(state):
    msg   = state['raw_message']
    corrected = correct_message_for_parsing(msg)
    mode  = detect_user_mode(corrected)
    info  = extract_symptoms(corrected)
    flags = detect_red_flags(corrected)
    if corrected != msg:
        print(f'[PARSE] Corrected spelling: {msg[:80]} -> {corrected[:80]}')
    print(f'[PARSE] Mode: {mode} | Symptoms: {info["symptoms"][:2]} | Red flags: {len(flags)}')
    return {**state, 'corrected_message': corrected, 'user_mode': mode, 'symptoms': info['symptoms'],
            'duration': info['duration'], 'age': info['age'],
            'medications': info['medications'], 'red_flags': flags, 'status': 'parsed',
            'workflow_path': state.get('workflow_path', []) + ['parse']}

def emergency_node(state):
    print('[EMERGENCY] Fast-tracking emergency response')
    msg = ('EMERGENCY - GO TO A&E IMMEDIATELY\n\n'
           'Call 112 (Nigeria Emergency) or go to nearest hospital now.\n'
           'Do NOT drive yourself.\n\n'
           'Nigeria Emergency: 112 | LASAMBUS Lagos: 08000-432584\n\n'
           'DISCLAIMER: This is not a substitute for professional medical care.')
    return {**state, 'urgency_level': 'emergency', 'urgency_score': 10,
            'triage_report': {'emergency_response': msg, 'patient_report': msg},
            'report_ready': True, 'status': 'emergency_complete',
            'workflow_path': state.get('workflow_path', []) + ['emergency']}

def acknowledge_node(state):
    mode = state.get('user_mode', 'patient')
    print(f'[ACK] {mode.upper()} | Ticket: {state["ticket_id"]}')
    return {**state, 'status': 'acknowledged',
            'workflow_path': state.get('workflow_path', []) + ['acknowledge']}

print('Sprint 1 nodes ready')

In [ ]:
# ── Sprint 2 nodes ────────────────────────────────────────────────────────────
RESEARCH_TOPICS = [
    'clinical presentation symptoms and diagnosis criteria',
    'WHO Nigeria treatment guidelines and protocols',
    'NHIS Nigeria coverage and treatment pathways',
    'differential diagnosis similar conditions to rule out',
    'recommended diagnostic tests and investigations',
    'drug treatment options dosages and contraindications',
    'self-care management and home treatment guidelines',
    'nutrition and dietary recommendations during illness',
]

def search_pubmed(query: str, max_results: int = 3) -> List[Dict]:
    try:
        r = requests.get(f'{PUBMED_BASE_URL}/esearch.fcgi',
            params={'db': 'pubmed', 'term': f'{query}[Title/Abstract]',
                    'retmax': max_results, 'retmode': 'json'}, timeout=8)
        ids = r.json().get('esearchresult', {}).get('idlist', [])
        if not ids: return []
        s = requests.get(f'{PUBMED_BASE_URL}/esummary.fcgi',
            params={'db': 'pubmed', 'id': ','.join(ids), 'retmode': 'json'}, timeout=8)
        result = s.json().get('result', {})
        return [{'pmid': pid, 'title': result.get(pid, {}).get('title', ''),
                 'year': result.get(pid, {}).get('pubdate', '')[:4],
                 'url': f'https://pubmed.ncbi.nlm.nih.gov/{pid}/'} for pid in ids if pid in result]
    except Exception as e:
        print(f'[PUBMED] Error: {e}')
        return []

def research_node(state):
    syms  = state.get('symptoms', [])
    msg   = state.get('raw_message', '')
    mode  = state.get('user_mode', 'patient')
    errors = list(state.get('errors', []))
    query = ', '.join(syms[:4]) if syms else msg[:150]
    print(f'[RESEARCH] Researching: {query[:60]}')
    results = []
    for i, topic in enumerate(RESEARCH_TOPICS, 1):
        prompt = f'Nigeria health context. Symptoms: {query}. Research: {topic}. Use WHO and NHIS guidelines.'
        print(f'  [{i}/{len(RESEARCH_TOPICS)}] {topic[:50]}')
        for attempt in range(3):
            try:
                resp = openai_client.responses.create(
                    model=RESEARCH_MODEL, tools=[{'type': 'web_search_preview'}], input=prompt)
                text = ''.join(c.text for block in resp.output
                               if hasattr(block, 'content') for c in block.content if hasattr(c, 'text'))
                if text.strip():
                    results.append(f'### {topic.upper()}\n{text.strip()}')
                    break
            except Exception as e:
                if attempt == 2: errors.append(f'Topic failed: {e}')
                else: time.sleep(2**attempt)
    articles = search_pubmed(f'{query} treatment Nigeria', 3)
    if articles:
        pb = 'PUBMED EVIDENCE\n' + '\n'.join([f'{a["title"]} ({a["year"]}) {a["url"]}' for a in articles])
        results.append(pb)
        print(f'[RESEARCH] PubMed: {len(articles)} articles')
    if not results:
        return {**state, 'status': 'error_research_failed',
                'errors': errors + ['All research failed'],
                'workflow_path': state.get('workflow_path', []) + ['research']}
    raw = f'MEDICAL RESEARCH: {query}\n\n' + '\n\n'.join(results)
    print(f'[RESEARCH] Done: {len(raw):,} chars, {len(results)} sections')
    return {**state, 'raw_research': raw, 'status': 'research_complete', 'errors': errors,
            'workflow_path': state.get('workflow_path', []) + ['research']}

def embed_and_store_node(state):
    research = state.get('raw_research', '')
    ticket   = state.get('ticket_id', 'UNKNOWN')
    syms     = state.get('symptoms', [])
    errors   = list(state.get('errors', []))
    words    = research.split()
    chunks   = [' '.join(words[i:i+200]) for i in range(0, len(words), 170) if words[i:i+200]]
    namespace = re.sub(r'[^a-z0-9-]', '', (syms[0] if syms else 'general').lower())[:30]
    namespace = f'{namespace}-{ticket[-8:].lower()}'
    print(f'[EMBED] {len(chunks)} chunks -> namespace: {namespace}')
    vectors, ids, valid = [], [], []
    for i in range(0, len(chunks), 10):
        batch = chunks[i:i+10]
        for attempt in range(3):
            try:
                emb = openai_client.embeddings.create(model=EMBED_MODEL, input=batch)
                for j, e in enumerate(emb.data):
                    cid = f'{ticket}-c{i+j:04d}'
                    vectors.append({'id': cid, 'values': e.embedding,
                        'metadata': {'ticket_id': ticket, 'text': batch[j][:1000],
                                     'source_type': 'web_search', 'chunk_idx': i+j}})
                    ids.append(cid); valid.append(batch[j])
                break
            except Exception as e:
                if attempt == 2: errors.append(str(e))
                else: time.sleep(2**attempt)
    if not vectors:
        return {**state, 'status': 'error_embed_failed', 'errors': errors,
                'workflow_path': state.get('workflow_path', []) + ['embed']}
    for i in range(0, len(vectors), 100):
        pinecone_index.upsert(vectors=vectors[i:i+100], namespace=namespace)
    print(f'[EMBED] {len(vectors)} vectors stored')
    return {**state, 'research_chunks': valid, 'pinecone_ids': ids, 'status': 'stored',
            'errors': errors, 'workflow_path': state.get('workflow_path', []) + ['embed']}

print('Sprint 2 nodes ready')

In [ ]:
# ── Sprint 3 nodes ────────────────────────────────────────────────────────────
PATIENT_TRIAGE_PROMPT = ('You are a medical triage AI for Nigeria. '
    'Analyse patient symptoms using ONLY provided clinical evidence. '
    'Return JSON: {"urgency_level": "emergency"|"urgent"|"standard"|"self-care", '
    '"urgency_score": 1-10, "differential": ["cond1","cond2","cond3"], '
    '"recommended_action": "action", "recommended_tests": ["test"], '
    '"self_care_eligible": true/false, "facility_needed": "home"|"clinic"|"hospital"|"emergency_room", '
    '"confidence": "low"|"medium"|"high"} '
    'Return ONLY valid JSON.')

CLINICIAN_TRIAGE_PROMPT = ('You are a clinical decision support AI for Nigerian healthcare providers. '
    'Return JSON: {"urgency_level": "emergency"|"urgent"|"standard"|"self-care", '
    '"urgency_score": 1-10, "differential": ["cond1 (ICD-10: X00)","cond2","cond3"], '
    '"recommended_action": "action", "recommended_tests": ["test1","test2"], '
    '"drug_recommendations": ["drug dose route"], "contraindications": [], '
    '"nhis_protocol": "protocol", "confidence": "low"|"medium"|"high"} '
    'Return ONLY valid JSON.')

def retrieve_node(state):
    syms  = state.get('symptoms', [])
    mode  = state.get('user_mode', 'patient')
    errors = list(state.get('errors', []))
    query = f'{"Clinical" if mode=="clinician" else "Patient"} Nigeria: {chr(44).join(syms[:4])}'
    print(f'[RETRIEVE] {query[:60]}')
    namespace = state.get('clinical_namespace')
    if not namespace:
        ticket = state.get('ticket_id', 'UNKNOWN')
        namespace = re.sub(r'[^a-z0-9-]', '', (syms[0] if syms else 'general').lower())[:30]
        namespace = f'{namespace}-{ticket[-8:].lower()}'
    fallback = [c for c in (state.get('research_chunks') or []) if isinstance(c, str) and c.strip()]
    if not fallback:
        raw_research = state.get('raw_research') or ''
        if raw_research.strip():
            words = raw_research.split()
            fallback = [' '.join(words[i:i+180]) for i in range(0, len(words), 160) if words[i:i+180]][:10]
    for attempt in range(3):
        try:
            emb = openai_client.embeddings.create(model=EMBED_MODEL, input=[query], timeout=30)
            res = pinecone_index.query(vector=emb.data[0].embedding, top_k=5, include_metadata=True, namespace=namespace)
            chunks = [m['metadata']['text'] for m in res['matches'] if m.get('metadata', {}).get('text')]
            if not chunks:
                if not fallback:
                    fallback = [query]
                print(f'[RETRIEVE] Pinecone returned no matches; using {len(fallback)} local research chunks')
                return {**state, 'retrieved_chunks': fallback[:10], 'status': 'retrieved_fallback',
                        'errors': errors + ['No Pinecone matches; used local research chunks'],
                        'workflow_path': state.get('workflow_path', []) + ['retrieve']}
            print(f'[RETRIEVE] {len(chunks)} chunks')
            return {**state, 'retrieved_chunks': chunks, 'status': 'retrieved', 'errors': errors,
                    'workflow_path': state.get('workflow_path', []) + ['retrieve']}
        except Exception as e:
            print(f'  Retrieve attempt {attempt+1}/3 failed: {e}')
            if not fallback:
                fallback = [query]
            return {**state, 'retrieved_chunks': fallback[:10], 'status': 'retrieved_fallback',
                    'errors': errors + [str(e), 'Used local research chunks fallback'],
                    'workflow_path': state.get('workflow_path', []) + ['retrieve']}
            time.sleep(2**attempt)

def rerank_node(state):
    syms   = state.get('symptoms', [])
    chunks = state.get('retrieved_chunks', [])
    errors = list(state.get('errors', []))
    query  = f'Nigeria clinical guidelines treatment for: {chr(44).join(syms[:4])}'
    print(f'[RERANK] {len(chunks)} chunks')
    for attempt in range(3):
        try:
            r = cohere_client.rerank(model='rerank-english-v3.0', query=query,
                                      documents=chunks, top_n=3, return_documents=True)
            reranked = [x.document.text for x in r.results if x.document]
            scores   = [round(x.relevance_score, 3) for x in r.results]
            print(f'[RERANK] Top-3 scores: {scores}')
            return {**state, 'reranked_chunks': reranked, 'status': 'reranked', 'errors': errors,
                    'workflow_path': state.get('workflow_path', []) + ['rerank']}
        except Exception as e:
            if attempt == 2:
                print('[RERANK] Fallback to top-3')
                return {**state, 'reranked_chunks': chunks[:3], 'status': 'reranked_fallback',
                        'errors': errors + [f'Cohere failed: {e}'],
                        'workflow_path': state.get('workflow_path', []) + ['rerank']}
            time.sleep(2**attempt)

def triage_node(state):
    syms    = state.get('symptoms', [])
    msg     = state.get('raw_message', '')
    mode    = state.get('user_mode', 'patient')
    chunks  = state.get('reranked_chunks') or state.get('retrieved_chunks') or []
    errors  = list(state.get('errors', []))
    context = '\n\n---\n\n'.join(chunks)
    prompt  = CLINICIAN_TRIAGE_PROMPT if mode == 'clinician' else PATIENT_TRIAGE_PROMPT
    user    = f'Context:\n{context}\n\nSymptoms: {syms}\nMessage: {msg}\nMode: {mode}'
    print(f'[TRIAGE] Mode: {mode}')
    for attempt in range(3):
        try:
            resp   = openai_client.chat.completions.create(
                model=RESEARCH_MODEL, temperature=0,
                response_format={'type': 'json_object'},
                messages=[{'role': 'system', 'content': prompt}, {'role': 'user', 'content': user}])
            parsed = json.loads(resp.choices[0].message.content)
            urgency = parsed.get('urgency_level', 'standard')
            score   = int(parsed.get('urgency_score', 5))
            diff    = parsed.get('differential') or []
            print(f'[TRIAGE] {urgency.upper()} ({score}/10) | {diff[:2]}')
            return {**state, 'urgency_level': urgency, 'urgency_score': score,
                    'differential': diff, 'status': 'triaged', 'errors': errors,
                    'triage_report': {'_classification': json.dumps(parsed)},
                    'workflow_path': state.get('workflow_path', []) + ['triage']}
        except Exception as e:
            if attempt == 2:
                return {**state, 'status': 'error_triage_failed',
                        'errors': errors + [str(e)],
                        'workflow_path': state.get('workflow_path', []) + ['triage']}
            time.sleep(2**attempt)

print('Sprint 3 nodes ready')

In [ ]:
# ── Sprint 4 nodes ────────────────────────────────────────────────────────────
URGENCY_EMOJIS  = {'emergency': 'RED', 'urgent': 'ORANGE', 'standard': 'YELLOW', 'self-care': 'GREEN'}
URGENCY_ACTIONS = {'emergency': 'Go to A&E IMMEDIATELY. Call 112 now.',
                   'urgent': 'See a doctor within 24 hours.',
                   'standard': 'Book a clinic appointment this week.',
                   'self-care': 'You can manage this at home with the guidance below.'}

def generate_report_node(state):
    urgency = state.get('urgency_level', 'standard')
    score   = state.get('urgency_score', 5)
    diff    = state.get('differential') or []
    syms    = state.get('symptoms', [])
    mode    = state.get('user_mode', 'patient')
    ticket  = state.get('ticket_id', '')
    chunks  = state.get('reranked_chunks') or state.get('retrieved_chunks') or []
    errors  = list(state.get('errors', []))
    context = '\n\n'.join(chunks[:3])
    action  = URGENCY_ACTIONS.get(urgency, 'See a doctor.')
    print(f'[REPORT] Generating {mode} report | {urgency.upper()}')
    if mode == 'clinician':
        sys_p = ('Clinical decision support for Nigerian healthcare. '
                 'Generate structured clinical report with ICD-10 codes, tests, drugs, NHIS protocol. '
                 'Base on provided context. Under 300 words.')
        user_p = f'Context:\n{context}\n\nPresentation: {syms}\n\nGenerate clinical triage report.'
    else:
        sys_p = ('Patient health assistant Nigeria. Write clear simple 150-word triage report. '
                 'No medical jargon. Base on provided context. Do not include disclaimer.')
        user_p = f'Context:\n{context}\n\nSymptoms: {syms}\nUrgency: {urgency}\nLikely conditions: {diff}'
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=REPORT_MODEL, temperature=0.2,
                messages=[{'role': 'system', 'content': sys_p}, {'role': 'user', 'content': user_p}])
            body = resp.choices[0].message.content.strip()
            full = (f'TRIAGE ASSESSMENT [{URGENCY_EMOJIS.get(urgency,"")}]\n'
                    f'Reference: {ticket}\nUrgency: {urgency.upper()} ({score}/10)\n\n'
                    f'ACTION: {action}\n\n{body}\n\n'
                    f'DISCLAIMER: This is not a substitute for professional medical advice. '
                    f'Always consult a qualified healthcare provider.')
            existing = state.get('triage_report') or {}
            key = 'clinician_report' if mode == 'clinician' else 'patient_report'
            existing[key] = full
            print(f'[REPORT] Generated ({len(full)} chars)')
            return {**state, 'triage_report': existing, 'status': 'report_generated',
                    'errors': errors, 'workflow_path': state.get('workflow_path', []) + ['report']}
        except Exception as e:
            if attempt == 2:
                return {**state, 'status': 'error_report_failed',
                        'errors': errors + [str(e)],
                        'workflow_path': state.get('workflow_path', []) + ['report']}
            time.sleep(2**attempt)

def deliver_node(state):
    mode      = state.get('user_mode', 'patient')
    urgency   = state.get('urgency_level', 'standard')
    report    = state.get('triage_report') or {}
    errors    = list(state.get('errors', []))
    key       = 'clinician_report' if mode == 'clinician' else 'patient_report'
    body      = report.get(key, report.get('emergency_response', 'Report unavailable'))
    if urgency == 'self-care':
        nutrition = get_nutrition_advice(state.get('symptoms', []), state.get('differential', []))
        body     += f'\n\nNIGERIAN NUTRITION ADVICE:\n{nutrition}'
        body     += '\n\nPROGRESS CHECK: You will receive a check-in in 24 hours.'
    print(f'\n{"="*60}')
    print(f'REPORT DELIVERED ({mode.upper()}):')
    print(f'{"="*60}')
    print(body[:600] + ('...' if len(body) > 600 else ''))
    print(f'{"="*60}')
    if TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID:
        try:
            import urllib.request
            for chunk in [body[i:i+4000] for i in range(0, len(body), 4000)]:
                payload = json.dumps({'chat_id': TELEGRAM_CHAT_ID, 'text': chunk}).encode()
                req = urllib.request.Request(
                    f'https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage',
                    data=payload, headers={'Content-Type': 'application/json'}, method='POST')
                urllib.request.urlopen(req)
                time.sleep(0.5)
            print('[DELIVER] Telegram sent successfully')
        except Exception as e:
            errors.append(f'Telegram failed: {e}')
    return {**state, 'report_ready': True, 'status': 'complete', 'errors': errors,
            'workflow_path': state.get('workflow_path', []) + ['deliver']}

def error_handler_node(state):
    print('PIPELINE ERROR - ALERT FIRED')
    for e in state.get('errors', []): print(f'  {e}')
    return {**state, 'status': 'failed',
            'workflow_path': state.get('workflow_path', []) + ['error_handler']}

print('Sprint 4 nodes ready')

## Build Full Pipeline Graph

In [ ]:
def route(state) -> str:
    return 'error_handler' if 'error' in state.get('status', '') else 'continue'

def route_after_parse(state) -> str:
    if state.get('red_flags'):
        return 'emergency'
    return 'acknowledge'

def build_full_pipeline():
    g = StateGraph(MedicalAgentState)
    nodes = [
        ('validate',      validate_input_node),
        ('parse',         parse_input_node),
        ('emergency',     emergency_node),
        ('acknowledge',   acknowledge_node),
        ('research',      research_node),
        ('embed',         embed_and_store_node),
        ('retrieve',      retrieve_node),
        ('rerank',        rerank_node),
        ('triage',        triage_node),
        ('report',        generate_report_node),
        ('deliver',       deliver_node),
        ('error_handler', error_handler_node),
    ]
    for name, fn in nodes:
        g.add_node(name, fn)

    g.set_entry_point('validate')
    g.add_conditional_edges('validate', route,
        {'continue': 'parse', 'error_handler': 'error_handler'})
    g.add_conditional_edges('parse', route_after_parse,
        {'emergency': 'emergency', 'acknowledge': 'acknowledge'})
    g.add_edge('emergency', 'deliver')
    g.add_conditional_edges('acknowledge', route,
        {'continue': 'research', 'error_handler': 'error_handler'})
    for step, nxt in [('research','embed'), ('embed','retrieve'), ('retrieve','rerank'),
                       ('rerank','triage'), ('triage','report'), ('report','deliver')]:
        g.add_conditional_edges(step, route,
            {'continue': nxt, 'error_handler': 'error_handler'})
    g.add_conditional_edges('deliver', route,
        {'continue': END, 'error_handler': 'error_handler'})
    g.add_edge('error_handler', END)
    return g.compile()

pipeline = build_full_pipeline()
print('Full pipeline compiled - 11 nodes')
print('Flow:')
print('  validate -> parse -> [emergency -> deliver]')
print('                    -> acknowledge -> research -> embed')
print('                       -> retrieve -> rerank -> triage -> report -> deliver -> END')

## LIVE DEMO - Run Full Pipeline

In [ ]:
# ========================================================
# LIVE DEMO - Change SYMPTOM_MESSAGE to test different cases
# ========================================================
SYMPTOM_MESSAGE = 'I have been having malaria fever, chills and headache for 3 days'
# Other test cases:
# 'I have typhoid fever, stomach pain and weakness for 5 days'
# 'I am a nurse. My patient has hypertension headache dizziness BP 180/110'
# 'I have chest pain and cannot breathe'  # <- triggers emergency

print('=' * 60)
print('AUTONOMOUS MEDICAL TRIAGE AGENT - LIVE RUN')
print(f'Message: {SYMPTOM_MESSAGE}')
print('=' * 60)

start = time.time()
initial = create_state(SYMPTOM_MESSAGE, channel='telegram')
final   = pipeline.invoke(initial)
elapsed = time.time() - start

print(f'\n{"-"*60}')
print(f'PIPELINE COMPLETE in {elapsed:.1f}s')
print(f'  Ticket      : {final.get("ticket_id", "unknown")}')
print(f'  Status      : {final.get("status", "unknown")}')
print(f'  Mode        : {final.get("user_mode", "patient")}')
urgency_level = final.get("urgency_level") or "unknown"
urgency_score = final.get("urgency_score")
urgency_score_text = f'{urgency_score}/10' if urgency_score is not None else 'unknown'
print(f'  Urgency     : {urgency_level} ({urgency_score_text})')
print(f'  Differential: {(final.get("differential") or [])[:2]}')
print(f'  Vectors     : {len(final.get("pinecone_ids") or [])}')
print(f'  Report ready: {final.get("report_ready", False)}')
print(f'  Path        : {" -> ".join(final.get("workflow_path") or [])}')
print(f'  Errors      : {final.get("errors", [])}')
print(f'{"-"*60}')

## Sprint 5 DoD Verification

In [ ]:
print('=' * 60)
print('SPRINT 5 - DEFINITION OF DONE CHECK')
print('=' * 60)

report   = final.get('triage_report') or {}
differential = final.get('differential') or []
workflow_path = final.get('workflow_path') or []
all_text = ' '.join(report.values())

checks = [
    ('Pipeline completed',        final['status'] in ('complete', 'emergency_complete')),
    ('Ticket ID generated',       bool(final.get('ticket_id', '').startswith('MED-'))),
    ('Mode detected',             final.get('user_mode') in ('patient', 'clinician')),
    ('Report generated',          bool(report.get('patient_report') or report.get('emergency_response'))),
    ('Safety disclaimer present', 'DISCLAIMER' in all_text),
    ('Urgency classified',        final.get('urgency_level') in ('emergency','urgent','standard','self-care')),
    ('Differential diagnosis',    len(differential) >= 1),
    ('8+ nodes executed',         len(workflow_path) >= 8),
]

all_pass = True
for label, ok in checks:
    print(f'  {"OK" if ok else "FAIL"} {label}')
    if not ok: all_pass = False

print(f'\n{"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')
print(f'Sprint 5 DoD: {"MET" if all_pass else "PARTIAL"}')

pathlib.Path('sprint5_final_output.json').write_text(
    json.dumps(dict(final), indent=2, default=str))
print('Final output saved to sprint5_final_output.json')

## QA - Test 3 Conditions

In [ ]:
# Quick QA test - 3 different conditions
qa_tests = [
    ('I have headache and mild fever for 1 day, feeling weak', 'patient'),
    ('I am a doctor. My patient has typhoid fever, abdominal pain, temperature 38.5', 'clinician'),
    ('I have chest pain and cannot breathe properly', 'patient'),  # emergency
]

print('QA TEST SUITE')
print('=' * 60)
qa_results = []

for msg, expected_mode in qa_tests:
    print(f'\nTesting: {msg[:55]}')
    r = pipeline.invoke(create_state(msg))
    disclaimer_ok = 'DISCLAIMER' in ' '.join((r.get('triage_report') or {}).values())
    status_ok     = r['status'] in ('complete', 'emergency_complete', 'failed')
    mode_ok       = r.get('user_mode') == expected_mode
    qa_results.append({'msg': msg[:40], 'status': r.get('status', 'unknown'),
                       'urgency': r.get('urgency_level'), 'mode': r.get('user_mode'),
                       'disclaimer': disclaimer_ok})
    print(f'  Status   : {r.get("status", "unknown")}')
    print(f'  Mode     : {r.get("user_mode")} ({"OK" if mode_ok else "FAIL"})')
    print(f'  Urgency  : {r.get("urgency_level")} ({r.get("urgency_score")}/10)')
    print(f'  Disclaimer: {"PRESENT" if disclaimer_ok else "MISSING - CRITICAL FAIL"}')

print('\n' + '='*60)
print('QA SUMMARY')
for r in qa_results:
    disc = 'OK' if r['disclaimer'] else 'FAIL'
    urgency = str(r.get('urgency') or 'unknown')
    mode = str(r.get('mode') or 'unknown')
    msg = str(r.get('msg') or '')
    print(f'  {disc} | {urgency} | {mode} | {msg}')
print('QA Complete')